# ForgeLM — Multi-Dataset Training

Mix multiple datasets with configurable ratios in a single training run.

**Requirements:** GPU runtime recommended (Runtime → Change runtime type → T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/multi_dataset.ipynb)

In [ ]:
# Step 1: Install ForgeLM
!pip install -q --no-cache-dir git+https://github.com/cemililik/ForgeLM.git bitsandbytes
!pip uninstall -y wandb -q
!forgelm --version

In [ ]:
# Step 2: Check GPU
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 3: Create two datasets
import json

# Dataset 1: General Q&A
general = [
    {
        "messages": [
            {"role": "user", "content": "What is AI?"},
            {
                "role": "assistant",
                "content": "AI is artificial intelligence — the simulation of human intelligence by machines.",
            },
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is ML?"},
            {"role": "assistant", "content": "ML is machine learning — a subset of AI where systems learn from data."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is deep learning?"},
            {
                "role": "assistant",
                "content": "Deep learning uses neural networks with many layers to learn complex patterns.",
            },
        ]
    },
]

# Dataset 2: Domain-specific (coding)
coding = [
    {
        "messages": [
            {"role": "user", "content": "Write hello world in Python"},
            {"role": "assistant", "content": "print('Hello, World!')"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a list?"},
            {
                "role": "assistant",
                "content": "A list is a mutable ordered sequence in Python, created with square brackets: my_list = [1, 2, 3]",
            },
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How to read a file?"},
            {"role": "assistant", "content": "Use: with open('file.txt') as f: content = f.read()"},
        ]
    },
]

for name, data in [("general.jsonl", general), ("coding.jsonl", coding)]:
    with open(name, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

print(f"Created: general.jsonl ({len(general)} samples), coding.jsonl ({len(coding)} samples)")

In [ ]:
# Step 4: Create config with multi-dataset mixing
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-1.7B-Instruct"
  max_length: 1024
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  output_dir: "./multi_checkpoints"
  num_train_epochs: 2
  per_device_train_batch_size: 2
  learning_rate: 2.0e-5

data:
  dataset_name_or_path: "general.jsonl"
  extra_datasets:
    - "coding.jsonl"
  mix_ratio: [0.7, 0.3]  # 70% general, 30% coding
"""

with open("multi_config.yaml", "w") as f:
    f.write(config_yaml)
print("Multi-dataset config saved! (70% general, 30% coding)")

In [ ]:
# Step 5: Validate
!forgelm --config multi_config.yaml --dry-run

In [ ]:
# Step 6: Train
!forgelm --config multi_config.yaml

In [ ]:
# Step 7: Verify output
import os

model_path = "./multi_checkpoints/final_model"
if not os.path.exists(model_path):
    print(f"Error: '{model_path}' not found. Check training output above.")
elif not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print(f"Error: adapter_config.json not found in '{model_path}'.")
else:
    print(f"Training completed! Model saved to: {model_path}")
    print(f"Files: {os.listdir(model_path)}")